# 超声波扫描测距：稳定演示版

按下面 1～5 的顺序手动运行。不要点击 Run All / 运行全部。本笔记本不配置开机自启动。

沿用已验证参数：中心指令 **84**（你校准的正前方）、指令范围 0～180、两轮扫描、每 3 个单位测距并更新显示、单位移动等待 0.03 秒、测量前稳定等待共 0.10 秒、目标渐隐 5 秒、原生 C SPI 加速。此前实测两轮 241 次测量、约 64.4 秒；这是参考值，不是保证值。

## 开始之前

- 先停止其他笔记本的扫描。不同笔记本共用同一块 FPGA，不能同时控制。
- **先关闭 MB102（舵机及超声波供电），PYNQ 保持开启。** 不要在舵机供电开启时重新加载 FPGA。
- 本流程按之前验证的冷启动条件使用：传感器安装牢固、线材无牵拉、机构未手动挪动且正前方参考未变。若实际姿态不明或安装变了，先检查，不要用本流程强行恢复未知姿态。
- 所有项目文件放在 `/home/xilinx/jupyter_notebooks/radar/`，本笔记本也上传到该文件夹。原有依赖不必重复上传。
- 后面只有供电阶段需要确认，没有端点逐次输入。命令寄存器读回不是实际角度反馈。
- 异常响声、卡住或牵拉时立即关闭 MB102；停止 Python **不会关闭 FPGA 的 PWM**。


## 1. 检查文件、导入驱动
这一格不访问硬件。重新运行会清除本笔记本的流程确认状态；不要在扫描过程中运行。


In [ ]:
from pathlib import Path
import hashlib
import importlib
import sys
import time

if globals().get('scanner') is not None and scanner.running:
    raise RuntimeError('先等待当前扫描结束，不要重新运行启动单元格。')

_prepared = False
_screen_ready = False
_power_ready = False
RADAR_DIR = Path('/home/xilinx/jupyter_notebooks/radar')
CENTER_COMMAND = 84
REQUIRED = (
    'radar_lcd.bit', 'radar_lcd.hwh',
    'st7796_lcd.py', 'radar_display.py', 'radar_partial_refresh.py',
    'radar_slow_scan.py', 'radar_fade_scan.py', 'radar_fast_scan.py',
    'echo_native_spi.py', 'echo_native_spi.c',
)
missing = [name for name in REQUIRED if not (RADAR_DIR / name).is_file()]
if missing:
    raise FileNotFoundError('请上传缺少的文件：' + ', '.join(missing))

# 固定本项目已经验证的 bit/hwh 配对，避免误用旧版本。
expected_hashes = {
    'radar_lcd.bit': 'd1a5c9bded7665fdc4f68c0217cbbb729b35974ac2c6224eea533c5e3b65c671',
    'radar_lcd.hwh': '30f843bb6ca57b6a3475e3fdab11c400a95afd690fa55270d3ddffeeb2ebdfa5',
}
for name, expected in expected_hashes.items():
    actual = hashlib.sha256((RADAR_DIR / name).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(name + ' 与已验证版本不一致；先核对文件，不要跳过检查。')

sys.path.insert(0, str(RADAR_DIR))
importlib.invalidate_caches()
for module_name in (
    'st7796_lcd', 'radar_display', 'radar_partial_refresh',
    'radar_slow_scan', 'radar_fade_scan', 'radar_fast_scan', 'echo_native_spi',
):
    module = importlib.import_module(module_name)
    if Path(module.__file__).resolve().parent != RADAR_DIR.resolve():
        raise RuntimeError('导入了其他目录的驱动：' + module_name)

from pynq import Overlay, Clocks, MMIO
from PIL import ImageDraw
from st7796_lcd import ST7796LCD
from radar_display import radar_background, rgb565_bytes, _font
from radar_fast_scan import CenteredFadingDisplay, IntervalRangefinder, CenteredFastScanner
from echo_native_spi import enable_native_spi

print('文件检查通过；尚未加载 FPGA、发送舵机指令或开始扫描。')


## 2. MB102 关闭时加载配置、准备中心指令
**先实际关闭 MB102，再运行并输入 `MB102已关闭`。** 本格会重新配置 FPGA，然后在舵机未供电时写入中心指令 84。写入 84 不代表此时测得了实际角度。


In [ ]:
if globals().get('scanner') is not None and scanner.running:
    raise RuntimeError('当前仍在扫描，禁止重新加载 FPGA。')
_prepared = False
_screen_ready = False
_power_ready = False
answer = input('实际关闭 MB102 后，输入 MB102已关闭 继续：').strip()
if answer != 'MB102已关闭':
    raise RuntimeError('已取消；没有加载 FPGA 或写入舵机指令。')

print('加载 FPGA 配置中，请保持 MB102 关闭……', flush=True)
ol = Overlay(str(RADAR_DIR / 'radar_lcd.bit'))
if not ol.is_loaded():
    raise RuntimeError('配置加载检查失败，请保持 MB102 关闭。')
required_ips = {'axi_gpio_0', 'axi_gpio_1', 'axi_gpio_2', 'axi_quad_spi_0'}
if not required_ips.issubset(ol.ip_dict):
    raise RuntimeError('配置中缺少所需 IP，请保持 MB102 关闭。')
if abs(float(Clocks.fclk0_mhz) - 125.0) > 0.5:
    raise RuntimeError('FCLK0 不是预期的 125 MHz，请保持 MB102 关闭并检查。')

servo_info = ol.ip_dict['axi_gpio_1']
servo_gpio = MMIO(servo_info['phys_addr'], servo_info['addr_range'])
servo_gpio.write(0x08, CENTER_COMMAND)
time.sleep(0.10)
if (servo_gpio.read(0x08) & 0xFF) != CENTER_COMMAND:
    raise RuntimeError('中心指令读回不一致，请保持 MB102 关闭。')
_prepared = True
print('配置已加载：', ol.is_loaded())
print('FCLK0：', Clocks.fclk0_mhz, 'MHz')
print('已准备中心指令：', CENTER_COMMAND)
print('继续保持 MB102 关闭，下一格初始化 LCD。')


## 3. 初始化 LCD、启用 C 加速、建立扫描对象
MB102 仍保持关闭。本格不发送舵机指令、不触发超声波测距。C 自检为 RAM 模型测试，不访问硬件；通过后才启用加速。若已有编译缓存，会复用它并再次自检。


In [ ]:
if not _prepared or _power_ready:
    raise RuntimeError('只在第 2 步完成且 MB102 仍关闭时运行本格。')
_screen_ready = False
if not ol.is_loaded() or (servo_gpio.read(0x08) & 0xFF) != CENTER_COMMAND:
    raise RuntimeError('配置或中心指令发生变化，请保持 MB102 关闭并检查。')

lcd = ST7796LCD(ol)
lcd.initialize()
enable_native_spi(lcd)

frame = radar_background(max_cm=120, ring_step_cm=30)
draw = ImageDraw.Draw(frame)
draw.rectangle((10, 45, 230, 64), fill=(2, 10, 7))
draw.text((13, 46), 'READY | CENTER CMD 84', font=_font(11), fill=(113, 170, 144))
pixels = rgb565_bytes(frame)
started = time.perf_counter()
lcd.show_rgb565(pixels)
lcd.backlight(True)
print(f'初始画面发送：{time.perf_counter() - started:.3f} 秒')

display = CenteredFadingDisplay(lcd, frame, center_command=CENTER_COMMAND, fade_seconds=5.0)
rangefinder = IntervalRangefinder(ol)
scanner = CenteredFastScanner(
    ol, display, rangefinder, center_command=CENTER_COMMAND, rounds=2,
    sample_stride=3, move_seconds=0.03, settle_seconds=0.10,
)
_screen_ready = True
print('LCD 和扫描对象准备完成；尚未开始扫描。')
print('确认画面正常，再按第 4 步打开 MB102。')


## 4. 打开 MB102，确认舵机在正前方稳定
确认 LCD 正常、安装牢固、线材无牵拉后，**现在手动打开 MB102**。观察舵机是否在你校准的正前方稳定保持，不能只看寄存器值。若异常，立即关闭 MB102，不要运行扫描。

确认正常后运行下格，输入 `上电稳定`。代码不发送运动指令。


In [ ]:
_power_ready = False
if not _prepared or not _screen_ready or scanner.running:
    raise RuntimeError('请先完成前面的准备步骤，并停止其他扫描。')
answer = input('MB102 已开启、实际正前方稳定且 LCD 正常，输入 上电稳定：').strip()
if answer != '上电稳定':
    raise RuntimeError('未确认，禁止开始扫描。若机构异常，请立即关闭 MB102。')
if not ol.is_loaded() or scanner.read_command() != CENTER_COMMAND:
    raise RuntimeError('配置或中心指令不匹配，未发送运动指令。')
_power_ready = True
print('准备完成。下一格开始两轮扫描。')


## 5. 开始两轮扫描（演示时重复运行这一格即可）
路径为指令 **84 → 0 → 180 → 0 → 180 → 84**，总共 720 次单位移动、241 次测量。每次移动只改变 1 个指令单位；不是在几个端点之间跳转。显示仍为水平半圆，保留中心校准。

仅在上次正常结束、仍稳定在正前方、期间未断电/重新配置/手动挪动时，才可只重复本格。过程中不要运行其他笔记本控制硬件。

中断或报错时不自动回中。若中途停止，不要直接改寄存器跳回 84，也不要直接重跑；先关闭 MB102 并检查实际姿态。按已验证的启动条件恢复后再从第 2 步准备。


In [ ]:
if not (_prepared and _screen_ready and _power_ready):
    raise RuntimeError('先按顺序完成 1～4 步；未发送扫描指令。')
if scanner.running:
    raise RuntimeError('当前扫描尚未结束。')
if scanner.display.lcd is not lcd or getattr(lcd, '_echo_native_backend', None) is None:
    raise RuntimeError('LCD 对象或原生加速状态不匹配，请停止并检查。')

try:
    started = time.perf_counter()
    completed = scanner.run()
    if not completed:
        raise RuntimeError('扫描未正常完成，请检查实际姿态；不会自动回中。')
    elapsed = time.perf_counter() - started
    print(f'扫描用时：{elapsed:.1f} 秒，测量次数：{scanner.samples_taken}')
    print('扫描正常完成，最后指令 84；现在仅更新画面渐隐，不再发送运动指令。')
    display.fade_out()
    print('渐隐结束。供电及机构状态未变时，可重复运行本格。')
except BaseException:
    _power_ready = False
    print('已停止，没有自动回中或关闭 PWM；机械异常时立即关闭 MB102。', flush=True)
    raise


## 演示结束与再次使用

1. 等扫描正常结束，确认不再移动。
2. **先关闭 MB102，再关闭 PYNQ。** 不要先断 PYNQ 而让舵机继续供电。
3. 保存笔记本用 Ctrl+S。运行输出可以保存，但内存变量和 FPGA 当前配置不会随笔记本保存。
4. 下次重新上电或重启内核后，从第 1 步重新按顺序操作，不要直接运行第 5 步。供电曾中断也必须重新确认实际机构状态。

注意：启动流程里的加载及中心指令步骤有意要求 MB102 关闭。此笔记本没有修改 FPGA 上电默认值，没有增加舵机关闭功能，也没有实现真实角度反馈。
